In [ ]:
from __future__ import annotations

import numpy as np
from pathlib import Path
import plotly.graph_objects as go
import torch
from torch import nn


"""
Interactive comparison of PCA vs. an autoencoder with a nonlinear encoder and linear decoder on a nonlinear 2D manifold in R^3.

- Data: curved 2D manifold embedded in 3D with noise and rotation.
- PCA: rank-2 subspace and reconstructions.
- Autoencoder: Encoder is MLP (nonlinear), Decoder is linear (no bias), trained with MSE on centered data.
- Plots: interactive 3D scatter (originals, PCA recon, AE recon) with PCA/AE planes, plus a 2D latent scatter.

Run: python this_file.py
Outputs: figs/nonlinear_encoder_linear_decoder/ae_3d.html, figs/nonlinear_encoder_linear_decoder/latent_2d.html
"""


def make_nonlinear_data(
    n: int = 1500, noise: float = 0.04, seed: int = 3
) -> np.ndarray:
    """
    Generate a nonlinear 2D manifold in R^3 using quadratic and sinusoidal components, then rotate and offset.

    Parameters
    ----------
    n : int
        Number of samples to generate.
    noise : float
        Standard deviation of additive Gaussian noise.
    seed : int
        Random seed.

    Returns
    -------
    np.ndarray
        Array of shape (n, 3) with dtype float32.
    """
    rng = np.random.default_rng(seed)
    u = rng.uniform(-2.5, 2.5, size=n).astype(np.float32)
    v = rng.uniform(-2.0, 2.0, size=n).astype(np.float32)
    x1 = u
    x2 = v + 0.15 * (u ** 2)
    x3 = 0.5 * (u ** 2) + 0.3 * np.sin(2.2 * v) + 0.2 * u * v
    X = np.stack([x1, x2, x3], axis=1).astype(np.float32)

    R = np.array(
        [[0.36, -0.80, 0.48],
         [0.80,  0.06, 0.60],
         [-0.48, 0.60, 0.64]],
        dtype=np.float32,
    )
    X = X @ R.T
    X += rng.normal(0.0, noise, size=X.shape).astype(np.float32)
    X += np.array([0.8, -0.4, 0.5], dtype=np.float32)
    return X


def pca_2d(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute the rank-2 PCA subspace and reconstructions.

    Parameters
    ----------
    X : np.ndarray
        Data of shape (n, 3).

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        mu: (3,) mean vector,
        W: (3,2) orthonormal principal directions,
        Z: (n,2) latent coordinates,
        X_hat: (n,3) reconstructions.
    """
    mu = X.mean(axis=0, dtype=np.float32)
    Xc = X - mu
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    V = Vt.T.astype(np.float32)
    W = V[:, :2].astype(np.float32)
    Z = Xc @ W
    X_hat = Z @ W.T + mu
    return mu, W, Z, X_hat


class NonlinearEncoderLinearDecoder(nn.Module):
    """
    Autoencoder with a nonlinear encoder and linear decoder (no bias).
    """

    def __init__(self, d_in: int = 3, d_latent: int = 2, h1: int = 96, h2: int = 96) -> None:
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(d_in, h1),
            nn.GELU(),
            nn.Linear(h1, h2),
            nn.GELU(),
            nn.Linear(h2, d_latent),
        )
        self.dec = nn.Linear(d_latent, d_in, bias=False)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Encode inputs to latent z and decode back to x-hat (both on centered coordinates).

        Parameters
        ----------
        x : torch.Tensor
            Centered inputs of shape (n, 3).

        Returns
        -------
        tuple[torch.Tensor, torch.Tensor]
            z: (n, 2),
            xhat: (n, 3) on centered coordinates.
        """
        z = self.enc(x)
        xhat = self.dec(z)
        return z, xhat


def train_ae_nonlinear_encoder_linear_decoder(
    X: np.ndarray,
    latent_dim: int = 2,
    epochs: int = 2000,
    lr: float = 1e-3,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Train the nonlinear-encoder/linear-decoder autoencoder with MSE on centered data.

    Parameters
    ----------
    X : np.ndarray
        Data of shape (n, 3).
    latent_dim : int
        Latent dimensionality.
    epochs : int
        Number of epochs.
    lr : float
        Learning rate.
    seed : int
        Torch random seed.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        mu: (3,) mean vector,
        W_plane: (3,2) orthonormal basis spanning the decoder plane,
        Z: (n,2) latent codes,
        X_hat: (n,3) reconstructions in original coordinates.
    """
    torch.manual_seed(seed)
    X_t = torch.tensor(X, dtype=torch.float32)
    mu = X_t.mean(dim=0, keepdim=True)
    Xc = X_t - mu

    model = NonlinearEncoderLinearDecoder(d_in=3, d_latent=latent_dim, h1=96, h2=96)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for _ in range(epochs):
        opt.zero_grad()
        z, xhat = model(Xc)
        loss = ((xhat - Xc) ** 2).mean()
        loss.backward()
        opt.step()

    with torch.no_grad():
        Z, Xc_hat = model(Xc)
        X_hat = (Xc_hat + mu).numpy().astype(np.float32)
        Z = Z.numpy().astype(np.float32)
        W_raw = model.dec.weight.detach().numpy()          # (3,2)
        Q, _ = np.linalg.qr(W_raw)
        W_plane = Q[:, :latent_dim].astype(np.float32)
        mu_np = mu.squeeze(0).numpy().astype(np.float32)

    return mu_np, W_plane, Z, X_hat


def make_plane(
    mu: np.ndarray, W: np.ndarray, Z: np.ndarray, steps: int = 40
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Build a grid in latent space and map to a plane in R^3 using x = mu + W z.

    Parameters
    ----------
    mu : np.ndarray
        Mean vector (3,).
    W : np.ndarray
        Basis (3,2).
    Z : np.ndarray
        Latents to set the plotting span (n,2).
    steps : int
        Grid resolution.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        U, V, Xs, Ys, Zs grids for Plotly Surface.
    """
    span = 2.2 * np.std(Z, axis=0)
    u = np.linspace(-span[0], span[0], steps, dtype=np.float32)
    v = np.linspace(-span[1], span[1], steps, dtype=np.float32)
    U, V = np.meshgrid(u, v)
    grid = np.stack([U.ravel(), V.ravel()], axis=1)
    P = grid @ W.T + mu
    Xs = P[:, 0].reshape(steps, steps)
    Ys = P[:, 1].reshape(steps, steps)
    Zs = P[:, 2].reshape(steps, steps)
    return U, V, Xs, Ys, Zs


def plot_interactive(
    X: np.ndarray,
    mu_pca: np.ndarray,
    W_pca: np.ndarray,
    Z_pca: np.ndarray,
    Xhat_pca: np.ndarray,
    mu_ae: np.ndarray,
    W_ae: np.ndarray,
    Z_ae: np.ndarray,
    Xhat_ae: np.ndarray,
) -> tuple[go.Figure, go.Figure]:
    """
    Build interactive 3D and 2D Plotly figures showing originals, reconstructions, and PCA/AE planes.

    Returns
    -------
    tuple[go.Figure, go.Figure]
        fig3d, fig2d.
    """
    Xo, Yo, Zo = X[:, 0], X[:, 1], X[:, 2]
    Xr, Yr, Zr = Xhat_pca[:, 0], Xhat_pca[:, 1], Xhat_pca[:, 2]
    _, _, Xs_p, Ys_p, Zs_p = make_plane(mu_pca, W_pca, Z_pca, steps=40)

    traces = [
        go.Scatter3d(x=Xo, y=Yo, z=Zo, mode="markers", name="Original", marker=dict(size=3, opacity=0.35)),
        go.Scatter3d(x=Xr, y=Yr, z=Zr, mode="markers", name="PCA recon", marker=dict(size=3, opacity=0.85)),
        go.Surface(x=Xs_p, y=Ys_p, z=Zs_p, name="PCA plane", showscale=False, opacity=0.25),
    ]

    Xa, Ya, Za = Xhat_ae[:, 0], Xhat_ae[:, 1], Xhat_ae[:, 2]
    traces.append(
        go.Scatter3d(x=Xa, y=Ya, z=Za, mode="markers", name="AE recon", marker=dict(size=3, opacity=0.85, symbol="diamond"))
    )
    _, _, Xs_a, Ys_a, Zs_a = make_plane(mu_ae, W_ae, Z_ae, steps=40)
    traces.append(go.Surface(x=Xs_a, y=Ys_a, z=Zs_a, name="AE plane", showscale=False, opacity=0.18))

    fig3d = go.Figure(data=traces)
    fig3d.update_layout(
        title="Nonlinear encoder + linear decoder vs PCA (R³)",
        scene=dict(xaxis_title="x₁", yaxis_title="x₂", zaxis_title="x₃", aspectmode="data"),
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=0, r=0, t=40, b=0),
    )

    z_traces = [go.Scatter(x=Z_pca[:, 0], y=Z_pca[:, 1], mode="markers", name="PCA latent z", marker=dict(size=5, opacity=0.85))]

    C = Z_ae.T @ Z_pca
    U_, _, Vt_ = np.linalg.svd(C, full_matrices=False)
    R = U_ @ Vt_
    Z_ae_aligned = Z_ae @ R
    z_traces.append(
        go.Scatter(
            x=Z_ae_aligned[:, 0],
            y=Z_ae_aligned[:, 1],
            mode="markers",
            name="AE latent z (aligned)",
            marker=dict(size=5, opacity=0.7, symbol="diamond"),
        )
    )

    fig2d = go.Figure(data=z_traces)
    fig2d.update_layout(
        title="Latent space (R²)",
        xaxis_title="z₁",
        yaxis_title="z₂",
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1,
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=0, r=0, t=40, b=0),
    )

    return fig3d, fig2d


def main() -> None:
    """
    Generate nonlinear data, compute PCA, train the nonlinear-encoder/linear-decoder AE, and render interactive plots.
    """
    X = make_nonlinear_data(n=2000, noise=0.05, seed=7)
    mu_pca, W_pca, Z_pca, Xhat_pca = pca_2d(X)
    mu_ae, W_ae, Z_ae, Xhat_ae = train_ae_nonlinear_encoder_linear_decoder(
        X, latent_dim=2, epochs=2500, lr=1e-3, seed=123
    )
    fig3d, fig2d = plot_interactive(X, mu_pca, W_pca, Z_pca, Xhat_pca, mu_ae, W_ae, Z_ae, Xhat_ae)

    outdir = Path("figs").joinpath("nonlinear_encoder_linear_decoder")
    outdir.mkdir(parents=True, exist_ok=True)
    fig3d.write_html(outdir.joinpath("ae_3d.html"), include_plotlyjs="cdn")
    fig2d.write_html(outdir.joinpath("latent_2d.html"), include_plotlyjs="cdn")
    fig3d.show()
    fig2d.show()


if __name__ == "__main__":
    main()